In [34]:
import os
import json
import datetime
import unicodedata
from google.colab import drive
import ipywidgets as w
from IPython.display import display, clear_output, HTML
# 1. Montar drive
drive.mount('/content/drive', force_remount=False)

# 2. Constante de ruta configurada
DIRECTORIO = '/content/drive/My Drive/LPII_2026'
EXTENSIONES_VALIDAS = ('.json', '.csv', '.pdf', '.txt', '.docx', '.xlsx')
FORMATO_FECHA = '%Y-%m-%d %H:%M:%S'

# Mapeo para los nombres legibles del ordenamiento
CLAVES_ORDEN = {
    'A a la Z': 'nombre',
    'Tamaño (KB)': 'tamano_kb',
    'Fecha de modificación': 'ultima_modificacion'
}

def normalizar(texto):
    texto = unicodedata.normalize('NFD', str(texto).lower())
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

def procesar_directorio_datos(directorio: str) -> dict:
    resultado_api = {
        "estado": "exitoso",
        "directorio": directorio,
        "timestamp_proceso": datetime.datetime.now().isoformat(),
        "archivos_procesados": [],
        "inventario": None
    }

    try:
        if not os.path.exists(directorio):
            raise FileNotFoundError(f"El directorio '{directorio}' no existe.")

        elementos = os.listdir(directorio)
        print(f"📁 Explorando directorio: {directorio}\n")

        for nombre in elementos:
            ruta_completa = os.path.join(directorio, nombre)

            if not os.path.isfile(ruta_completa):
                continue

            _, extension = os.path.splitext(nombre)
            extension = extension.lower()

            if extension not in EXTENSIONES_VALIDAS:
                continue

            tamano_bytes = os.path.getsize(ruta_completa)
            tamano_kb = round(tamano_bytes / 1024, 2)

            mtime_timestamp = os.path.getmtime(ruta_completa)
            fecha_modificacion = datetime.datetime.fromtimestamp(mtime_timestamp).strftime(FORMATO_FECHA)

            info_archivo = {
                "nombre": nombre,
                "nombre_norm": normalizar(nombre),
                "extension": extension,
                "tamano_kb": tamano_kb,
                "ultima_modificacion": fecha_modificacion
            }

            resultado_api["archivos_procesados"].append(info_archivo)
            print(f" └── [Archivo Válido] {nombre} ({tamano_kb} KB) - Modificado: {fecha_modificacion}")

            if nombre.lower() == "inventario.json":
                try:
                    with open(ruta_completa, 'r', encoding='utf-8') as f:
                        contenido_json = json.load(f)
                        if isinstance(contenido_json, list):
                            resultado_api["inventario"] = contenido_json
                            print(f"     ✅ 'inventario.json' parseado correctamente (Registros: {len(contenido_json)})")
                        else:
                            resultado_api["inventario"] = {"error": "Formato inválido: se esperaba una lista"}
                except json.JSONDecodeError:
                    resultado_api["inventario"] = {"error": "JSON malformado"}

    except Exception as e:
        resultado_api["estado"] = "error"
        resultado_api["mensaje"] = str(e)

    return resultado_api


# 3. Algoritmo QuickSort implementado desde cero
def quick_sort_archivos(lista, clave, reverse=False):
    if len(lista) <= 1:
        return lista

    pivote = lista[len(lista) // 2]
    left, middle, right = [], [], []

    for x in lista:
        val_x = x[clave]
        val_piv = pivote[clave]

        if isinstance(val_x, str):
            val_x = val_x.lower()
            val_piv = val_piv.lower()

        if val_x < val_piv:
            left.append(x)
        elif val_x > val_piv:
            right.append(x)
        else:
            middle.append(x)

    if reverse:
        return quick_sort_archivos(right, clave, reverse=True) + middle + quick_sort_archivos(left, clave, reverse=True)
    else:
        return quick_sort_archivos(left, clave, reverse=False) + middle + quick_sort_archivos(right, clave, reverse=False)


# 4. Renderizado HTML con paleta Violeta Neón y filas alternadas (Zebra Striping)
def formatear_html(archivos_filtrados, total_encontrados):
    if total_encontrados == 0:
        return """
        <div style="padding: 40px 20px; background: rgba(239, 68, 68, 0.08); color: #f87171; border-radius: 16px; border: 1px solid rgba(239, 68, 68, 0.2); font-family: 'Inter', sans-serif; text-align: center; margin-top: 15px;">
            <div style="font-size: 28px; margin-bottom: 6px;">🔍</div>
            <h4 style="margin: 0 0 6px 0; font-size: 16px; font-weight: 600;">Sin resultados</h4>
            <p style="margin: 0; font-size: 13px; color: #94a3b8;">Intenta ajustar los parámetros de búsqueda o limpiar los filtros.</p>
        </div>
        """

    html = f"""
    <div style="font-family: 'Inter', sans-serif; margin-top: 15px;">
        <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 16px; background: #131b2e; padding: 14px 20px; border-radius: 14px; border: 1px solid #1e293b; font-size: 13px; color: #94a3b8;">
            <span>📊 Total de archivos encontrados:</span>
            <span style="background: #8b5cf6; color: #ffffff; padding: 3px 12px; border-radius: 20px; font-weight: 700; font-size: 13px;">{total_encontrados}</span>
        </div>
        <div style="overflow-x: auto;">
            <table style="width: 100%; border-collapse: collapse; background: #131b2e; border-radius: 12px; overflow: hidden; border: 1px solid #1e293b; color: #ffffff; font-size: 13px;">
                <thead>
                    <tr style="background: #1b2640; text-align: left; border-bottom: 1px solid #1e293b;">
                        <th style="padding: 12px 16px;">📁 Nombre del Archivo</th>
                        <th style="padding: 12px 16px;">🏷️ Extensión</th>
                        <th style="padding: 12px 16px;">💾 Tamaño (KB)</th>
                        <th style="padding: 12px 16px;">🕒 Última Modificación</th>
                    </tr>
                </thead>
                <tbody>
    """

    for i, a in enumerate(archivos_filtrados):
        ext = a['extension']

        if ext == '.json':
            icono, badge_bg, badge_col = "📋", "rgba(139, 92, 246, 0.15)", "#a78bfa"
        elif ext == '.csv':
            icono, badge_bg, badge_col = "📊", "rgba(56, 189, 248, 0.15)", "#38bdf8"
        elif ext == '.pdf':
            icono, badge_bg, badge_col = "📕", "rgba(248, 113, 113, 0.15)", "#f87171"
        elif ext == '.txt':
            icono, badge_bg, badge_col = "📝", "rgba(156, 163, 175, 0.15)", "#9ca3af"
        elif ext == '.docx':
            icono, badge_bg, badge_col = "📘", "rgba(236, 72, 153, 0.15)", "#f472b6"
        elif ext == '.xlsx':
            icono, badge_bg, badge_col = "🟩", "rgba(52, 211, 153, 0.15)", "#34d399"
        else:
            icono, badge_bg, badge_col = "📄", "rgba(255, 255, 255, 0.1)", "#ffffff"

        # Filas alternadas (Zebra Striping)
        bg_fila = "#131b2e" if i % 2 == 0 else "#192237"

        html += f"""
                    <tr style="border-bottom: 1px solid #1e293b; background-color: {bg_fila};">
                        <td style="padding: 12px 16px; word-break: break-all;">{icono} {a['nombre']}</td>
                        <td style="padding: 12px 16px;"><span style="background: {badge_bg}; color: {badge_col}; padding: 3px 8px; border-radius: 6px; font-family: monospace; font-weight: 600;">{ext}</span></td>
                        <td style="padding: 12px 16px;">{a['tamano_kb']} KB</td>
                        <td style="padding: 12px 16px; color: #94a3b8;">{a['ultima_modificacion']}</td>
                    </tr>
        """

    html += """
                </tbody>
            </table>
        </div>
    </div>
    """
    return html


# 5. Interfaz con ipywidgets, membrete separado y paleta Violeta Neón
def lanzar_interfaz(directorio=DIRECTORIO):
    datos_iniciales = procesar_directorio_datos(directorio)
    archivos_totales = datos_iniciales["archivos_procesados"]

    if datos_iniciales["estado"] == "error":
        print(f"❌ Error al cargar el directorio: {datos_iniciales.get('mensaje')}")
        return

    display(HTML("""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
        .widget-box, .widget-inline-flex { font-family: 'Inter', sans-serif !important; }
        .widget-text input, .widget-dropdown select {
            background-color: #1b2640 !important;
            color: #ffffff !important;
            border: 1px solid #1e293b !important;
            border-radius: 10px !important;
            padding: 8px 12px !important;
            font-size: 13px !important;
            height: auto !important;
            box-sizing: border-box !important;
        }
        .widget-dropdown select { padding-right: 28px !important; }
        .widget-text input:focus, .widget-dropdown select:focus {
            border-color: #8b5cf6 !important;
            outline: none !important;
        }
    </style>
    """))

    def crear_label(texto):
        return w.HTML(f'<span style="font-family: \'Inter\', sans-serif; font-size: 12px; font-weight: 500; color: #94a3b8; margin-bottom: 6px; display: block;">{texto}</span>')

    txt_buscar = w.Text(placeholder='Escribe parte del nombre...', layout=w.Layout(width='100%'))
    dd_tipo = w.Dropdown(options=['Todos'] + list(EXTENSIONES_VALIDAS), value='Todos', layout=w.Layout(width='100%'))
    dd_orden = w.Dropdown(options=list(CLAVES_ORDEN), value='A a la Z', layout=w.Layout(width='100%'))

    boton = w.Button(
        description=' FILTRAR ',
        layout=w.Layout(width='100%', height='42px', margin='16px 0 0 0')
    )
    boton.style.button_color = '#8b5cf6'

    salida = w.Output(layout=w.Layout(width='100%'))

    def ejecutar(_=None):
        criterio_texto = normalizar(txt_buscar.value)
        filtrados = [a for a in archivos_totales if criterio_texto in a['nombre_norm']]

        if dd_tipo.value != 'Todos':
            filtrados = [a for a in filtrados if a['extension'] == dd_tipo.value]

        clave_seleccionada = CLAVES_ORDEN[dd_orden.value]
        revertir = True if dd_orden.value in ['Tamaño (KB)', 'Fecha de modificación'] else False

        archivos_ordenados = quick_sort_archivos(filtrados, clave_seleccionada, reverse=revertir)

        with salida:
            clear_output()
            display(HTML(formatear_html(archivos_ordenados, len(archivos_ordenados))))

    boton.on_click(ejecutar)

    # Panel de control con membrete separado y diseño Violeta Neón
    panel_controles = w.VBox([
        w.HTML("""
            <div>
                <h2 style='margin: 0 0 10px 0; font-size: 18px; color: #ffffff; font-weight: 700;'> EXPLORADOR DE ARCHIVOS DE GOOGLE DRIVE</h2>
                <p style='margin: 0 0 18px 0; font-size: 13px; color: #94a3b8; line-height: 1.5;'>
                    Es un buscador interactivo para explorar metadatos desde Google Drive, filtrar por formato y ordenar elementos de forma eficiente mediante el algoritmo <b>QuickSort</b>.
                </p>
            </div>
        """),
        w.HBox([
            w.VBox([crear_label("🔍 Buscador de Archivos"), txt_buscar], layout=w.Layout(flex='2')),
            w.VBox([crear_label("📂 Filtrar por Extensión"), dd_tipo], layout=w.Layout(flex='1')),
            w.VBox([crear_label("⚡ Ordenar por "), dd_orden], layout=w.Layout(flex='1'))
        ], layout=w.Layout(gap='12px', width='100%')),
        boton
    ], layout=w.Layout(
        backgroundColor='#131b2e',
        padding='20px',
        borderRadius='16px',
        border='1px solid #1e293b',
        boxShadow='0 10px 15px -3px rgba(0, 0, 0, 0.4)',
        margin='0 0 16px 0'
    ))

    app = w.VBox([panel_controles, salida], layout=w.Layout(
        backgroundColor='#0b0f19',
        padding='16px',
        borderRadius='20px',
        width='100%'
    ))

    display(app)
    ejecutar()

if __name__ == '__main__':
    lanzar_interfaz()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Explorando directorio: /content/drive/My Drive/LPII_2026

 └── [Archivo Válido] Calculadora.csv (1.77 KB) - Modificado: 2026-10-27 16:07:47
 └── [Archivo Válido] CuentaRegresiva.json (1.29 KB) - Modificado: 2026-10-27 16:09:25
 └── [Archivo Válido] inventario (1).json (3.94 KB) - Modificado: 2026-10-27 16:24:28
 └── [Archivo Válido] inventario.json (0.28 KB) - Modificado: 2026-10-27 16:24:41
     ✅ 'inventario.json' parseado correctamente (Registros: 3)
